In [1]:
import pandas as pd
import os

In [2]:
tracker_path = '../../../outputs/trackers/demand_tracker.xlsx'

In [3]:
sp_path = os.environ['SP_BASE_PATH']
sp_path

'C:/Users/mpy/OneDrive - Exus Management Partners/EU - Strategy & Markets - Documentos'

In [4]:
df_raw = pd.read_excel(tracker_path)

df_raw = df_raw[df_raw['Source'] == 'Aurora']

In [5]:
from common_libs.utils.utils_dates import map_month_abbr

df_raw['Month'] = df_raw['release_month'].apply(map_month_abbr)

df_raw['Date'] = pd.to_datetime(df_raw['release_year'].astype(int).astype(str) + '-' + df_raw['Month'].astype(str), format='%Y-%m')

In [6]:
df_raw

,variable,units,name,currency,sensitivity,region_code,Source,Scenario,country,region,...,2053,2054,2055,2056,2057,2058,2059,2060,Month,Date
0,Data centre demand,TWh,France Oct 25 (Central),EUR,central,fra,Aurora,Central,France,France,...,18700.0,18700.0,18800.0,18800.0,18900.0,18900.0,19000.0,19100.0,1,2026-01-01
1,Data centre demand,TWh,France Q1 26 (Central),EUR,central,fra,Aurora,Central,France,France,...,18700.0,18700.0,18800.0,18800.0,18900.0,18900.0,19000.0,19100.0,1,2026-01-01
2,Data centre demand,TWh,France Q2 26 (Central),EUR,central,fra,Aurora,Central,France,France,...,37900.0,38000.0,38200.0,38400.0,38400.0,38600.0,38700.0,38900.0,1,2026-01-01
3,EV demand,TWh,France Apr 2021 (Central),EUR,central,fra,Aurora,Central,France,France,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,2026-01-01
4,EV demand,TWh,France Apr 2022 (Central),EUR,central,fra,Aurora,Central,France,France,...,82500.0,83400.0,84300.0,85200.0,86000.0,86800.0,87500.0,88300.0,1,2026-01-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3365,Total demand,TWh,Iberia Oct 24 (Low),EUR,low,esp,Aurora,Low,Spain,Spain,...,386500.0,391000.0,395600.0,400600.0,404400.0,408500.0,412700.0,418100.0,1,2026-01-01
3366,Total demand,TWh,Iberia Oct 25 (Low),EUR,low,esp,Aurora,Low,Spain,Spain,...,391800.0,396300.0,402300.0,407500.0,411300.0,415200.0,418800.0,422500.0,1,2026-01-01
3367,Total demand,TWh,Iberia October 2020 (Low),EUR,low,esp,Aurora,Low,Spain,Spain,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,2026-01-01
3368,Total demand,TWh,Iberia October 2021 (Low),EUR,low,esp,Aurora,Low,Spain,Spain,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,2026-01-01


In [8]:
df_raw.sort_values(by=['region', 'Scenario', 'variable', 'Date'], inplace=True)

df_raw_fil = df_raw[
    (df_raw['units'] == 'TWh') & 
    (df_raw['Scenario'] == 'Central')
    ].reset_index(drop=True)
type(df_raw_fil)
df_raw_fil

,variable,units,name,currency,sensitivity,region_code,Source,Scenario,country,region,...,2053,2054,2055,2056,2057,2058,2059,2060,Month,Date
0,Data centre demand,TWh,France Oct 25 (Central),EUR,central,fra,Aurora,Central,France,France,...,18700.0,18700.0,18800.0,18800.0,18900.0,18900.0,19000.0,19100.0,1,2026-01-01
1,Data centre demand,TWh,France Q1 26 (Central),EUR,central,fra,Aurora,Central,France,France,...,18700.0,18700.0,18800.0,18800.0,18900.0,18900.0,19000.0,19100.0,1,2026-01-01
2,Data centre demand,TWh,France Q2 26 (Central),EUR,central,fra,Aurora,Central,France,France,...,37900.0,38000.0,38200.0,38400.0,38400.0,38600.0,38700.0,38900.0,1,2026-01-01
3,EV demand,TWh,France Apr 2021 (Central),EUR,central,fra,Aurora,Central,France,France,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,2026-01-01
4,EV demand,TWh,France Apr 2022 (Central),EUR,central,fra,Aurora,Central,France,France,...,82500.0,83400.0,84300.0,85200.0,86000.0,86800.0,87500.0,88300.0,1,2026-01-01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1040,Total demand,TWh,Iberia Oct 24 (Central),EUR,central,esp,Aurora,Central,Spain,Spain,...,461300.0,466900.0,472000.0,477100.0,481900.0,486800.0,490900.0,496200.0,1,2026-01-01
1041,Total demand,TWh,Iberia Oct 25 (Central),EUR,central,esp,Aurora,Central,Spain,Spain,...,466900.0,472500.0,477700.0,482400.0,487100.0,492100.0,496300.0,502200.0,1,2026-01-01
1042,Total demand,TWh,Iberia October 2020 (Central),EUR,central,esp,Aurora,Central,Spain,Spain,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,2026-01-01
1043,Total demand,TWh,Iberia October 2021 (Central),EUR,central,esp,Aurora,Central,Spain,Spain,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1,2026-01-01


In [9]:

# We drop columns that have at least one missing value, as we will later perform the variation
# of average price between consecutive forecasts. This way we ensure that the time periods we compare are the same across all forecasts.

# We now include a column with the average price across all technologies, to be used as a reference in the analysis.
# Take into account that year data are columns, so we need to specify axis=1 when calculating the mean.

# We first calclulate year period, based on columns

year_cols = df_raw_fil.columns[df_raw_fil.columns.str.contains(r'20\d{2}')]

# Coerce year columns to numeric — Excel separator strings like '---' become NaN
df_raw_fil[year_cols] = df_raw_fil[year_cols].apply(pd.to_numeric, errors='coerce')

df_final = df_raw_fil.drop(columns=year_cols)

df_final[year_cols] = df_final.groupby(['Country', 'Scenario', 'variable'])[year_cols].diff()
# drop first row of each group, as it will have NaN value in the variation column
# df_final = df_final.dropna(subset=['variation of average price'], axis=1)
# Now difference in percentage
df_final[year_cols] = df_final.groupby(['Country', 'Scenario', 'variable'])[year_cols].pct_change() * 100

# df_final.to_excel('processed_prices.xlsx')

df_final

ValueError: Cannot mask with non-boolean array containing NA / NaN values

In [ ]:
df_final

In [9]:
dates_df = df_final[['Date']].dropna().drop_duplicates().sort_values(by='Date').reset_index(drop=True)
last_4_dates = dates_df['Date'].unique()[-4:]
last_4_dates

<DatetimeArray>
['2025-07-01 00:00:00', '2025-10-01 00:00:00', '2026-01-01 00:00:00',
 '2026-04-01 00:00:00']
Length: 4, dtype: datetime64[ns]

In [12]:

final_dict_deltas = {}
final_dict_abs = {}
dates_df = df_final[['Date']].dropna().drop_duplicates().sort_values(by='Date').reset_index(drop=True)
last_4_dates = dates_df['Date'].unique()[-4:]

# Get Release date labels in chronological order (Date is already sorted ascending)
ordered_release_dates = (
    df_final[df_final['Date'].isin(last_4_dates)]
    .sort_values(by='Date')
    [['Date', 'Release date']]
    .drop_duplicates(subset='Date')
    ['Release date']
    .tolist()
)

for technology in df_final['Technology view'].unique():
    df_tech = df_final[(df_final['Technology view'] == technology) & (df_final['Date'].isin(last_4_dates))].sort_values(by='Date', ascending=False).reset_index(drop=True)
    df_tech = df_tech[['Release date', 'Country', 'Average price', 'variation of average price']].reset_index(drop=True)

    df_abs_pivot = df_tech.pivot(index='Country', columns='Release date', values='Average price')[ordered_release_dates]
    final_dict_abs[technology] = df_abs_pivot

    df_deltas_pivot = df_tech.pivot(index='Country', columns='Release date', values='variation of average price')[ordered_release_dates]
    final_dict_deltas[technology] = df_deltas_pivot

final_dict_abs['Baseload'].columns.to_list()


['2025-Jul', '2025-Oct', '2026-Jan', '2026-Apr']

In [67]:
import json

# Assuming you have two dicts — one for series values, one for deltas
# series_dict = {"Onshore wind": df_series, "Solar PV": df_series_2, ...}
# deltas_dict = {"Onshore wind": df_deltas, "Solar PV": df_deltas_2, ...}

SERIES = {}
for technology, df in final_dict_abs.items():
    SERIES[technology] = {
        country: row.tolist()
        for country, row in df.iterrows()
    }

COUNTRIES = df_final['Country'].unique().tolist()

DELTAS = {}
for technology, df in final_dict_deltas.items():
    DELTAS[technology] = {
        country: row.tolist()
        for country, row in df.iterrows()
    }

with open('input.html', 'r') as f:
    template = f.read()

template = template.replace('{{SERIES}}', json.dumps(SERIES))
template = template.replace('{{DELTAS}}', json.dumps(DELTAS))
template = template.replace('{{COUNTRIES}}', json.dumps(COUNTRIES))

with open('output.html', 'w') as f:
    f.write(template)

In [14]:

import json

# --- dynamic pieces ---
SERIES = {}
for technology, df in final_dict_abs.items():
    SERIES[technology] = {
        country: row.tolist()
        for country, row in df.iterrows()
    }

COUNTRIES = df_final['Country'].unique().tolist()

DELTAS = {}
for technology, df in final_dict_deltas.items():
    DELTAS[technology] = {
        country: row.tolist()
        for country, row in df.iterrows()
    }
    
COUNTRIES = df_final['Country'].unique().tolist()

# Auto-derive releases from any DataFrame
RELEASES  = final_dict_abs['Baseload'].columns.to_list()
LATEST    = RELEASES[-1]

# Auto-derive codes
preferred_country_order = ['Spain', 'Italy', 'Germany', 'Poland', 'GB', 'Portugal', 'Ireland', 'France']
COUNTRIES = [c for c in preferred_country_order if c in COUNTRIES]

CODES_MAP = {
    'Spain': 'ESP',
    'Italy': 'IT',
    'Germany': 'DE',
    'Poland': 'PL',
    'GB': 'UK',
    'Portugal': 'PT',
    'Ireland': 'IE',
    'France': 'FR',
}
CODES     = {c: CODES_MAP.get(c, c[:2].upper()) for c in COUNTRIES}

# Colors matched to legend image (order: Spain, Italy, Germany, Poland, GB, Portugal, Ireland, France)
PALETTE   = ['#D4A017', '#009AA8', '#1A3C2A', '#888888', '#005F6C', '#90BE50', '#C0C0C0', '#A8D8E0']
COLORS    = {c: PALETTE[i % len(PALETTE)] for i, c in enumerate(COUNTRIES)}

# --- fill template ---
with open('input.html') as f:
    template = f.read()

for key, val in [
    ('{{SERIES}}',       json.dumps(SERIES)),
    ('{{DELTAS}}',       json.dumps(DELTAS)),
    ('{{COUNTRIES}}',    json.dumps(COUNTRIES)),
    ('{{RELEASES}}',     json.dumps(RELEASES)),
    ('{{CODES}}',        json.dumps(CODES)),
    ('{{COLORS}}',       json.dumps(COLORS)),
    ('{{LATEST_LABEL}}', LATEST),
]:
    template = template.replace(key, val)

with open('output.html', 'w') as f:
    f.write(template)


In [65]:
DELTAS

{'Onshore wind': {'France': [-0.5129719444444447,
   1.528605325396839,
   -0.9150994285714233,
   0.5545319999999947],
  'GB': [-0.5985691964527575,
   -1.5940356869111838,
   -0.6150439390857301,
   1.964768844188562],
  'Germany': [-0.507663888888878,
   -0.9190053412698376,
   -0.13623057142857675,
   -1.9401199999999932],
  'Ireland': [0.7455683333333241,
   1.3213506190476068,
   -0.16562571428571005,
   1.4516834285714566],
  'Italy': [0.15509249999998076,
   1.3216964285714425,
   -0.35101028571428117,
   3.397890285714297],
  'Poland': [-0.6363171719359997,
   2.5785476713632676,
   0.26244649736230485,
   -2.073910085381982],
  'Portugal': [-0.7775536111111023,
   0.8720017777777969,
   -0.14220400000000666,
   nan],
  'Spain': [-0.7537249999999602,
   -1.9072622539682698,
   -0.14218971428570626,
   nan]},
 'Solar PV': {'France': [-0.35553464717952465,
   -1.210978785714289,
   -0.13460742857143515,
   0.10533371428572735],
  'GB': [-0.6498177062777728,
   -1.989367874565147